In [2]:
import pandas as pd

In [3]:
sound = pd.read_csv('sound.csv')

In [4]:
sound.shape

(657, 4)

In [5]:
sound = sound.drop(columns={'Unnamed: 3'})

In [6]:
sound.shape

(657, 3)

In [7]:
sound = sound.rename(columns={"City": "city"})
sound["city"] = sound["city"].str.strip()

In [8]:
sound.Tour.unique()
sound["city"] = sound["city"].replace("Montreal", "Montréal")
sound["city"] = sound["city"].replace("Aukland", "Auckland")

In [9]:
sound.shape

(657, 3)

In [10]:
cities = pd.read_csv("worldcities.csv")  # columns: city, country, lat, lng, population...

cities_small = (
    cities[["city", "country", "lat", "lng", "population"]]
    .sort_values("population", ascending=False)
    .drop_duplicates(subset="city", keep="first")
)


In [11]:

n = len(sound)             # should be 657
sound = sound.merge(cities_small, on="city", how="left", validate="m:1")
assert len(sound) == n
sound.shape    

(657, 7)

In [12]:
sound.isna().sum()

Date           0
city           0
Tour           0
country       62
lat           62
lng           62
population    62
dtype: int64

In [13]:
sound.shape

(657, 7)

In [14]:
unmatched = sound[sound["lat"].isna()]
print(unmatched[["city", "country"]].to_string())

                    city country
64           Wallingford     NaN
65           Stroudsburg     NaN
70      Lake Buena Vista     NaN
92          Southhampton     NaN
93       Southend On Sea     NaN
141        Frederiksberg     NaN
153         Santa Barbra     NaN
154             Berekley     NaN
157             Portalnd     NaN
178             Nurnberg     NaN
179             Nurnberg     NaN
180             Nurnberg     NaN
186      Seaside Heights     NaN
189               Pilton     NaN
190           Norrkoping     NaN
191             Werchter     NaN
192               Gdansk     NaN
193       Hradec Kralove     NaN
196         Auchterarder     NaN
197               Roeser     NaN
199             Tonsberg     NaN
201               Orpesa     NaN
202                Byron     NaN
208    Genting Highlands     NaN
211                Osaka     NaN
215           Stradbally     NaN
219            Sao Paulo     NaN
222             Flushing     NaN
223        Benito Juarez     NaN
224       

In [15]:
sound.shape

(657, 7)

In [16]:
unmatched.to_csv('unmatched.csv')

In [17]:
sound.head()

,Date,city,Tour,country,lat,lng,population
0,9/5/2014,Kansas City,The 1975 World Tour,United States,39.1238,-94.5541,1714910.0
1,9/6/2014,St. Louis,The 1975 World Tour,United States,38.6359,-90.2451,2129918.0
2,9/7/2014,Boston,The 1975 World Tour,United States,42.3188,-71.0852,4453352.0
3,9/14/2014,Shrewsbury,The 1975 World Tour,United Kingdom,52.7080,-2.7540,71715.0
4,9/21/2014,Manchester,The 1975 World Tour,United Kingdom,53.4790,-2.2452,547627.0


In [18]:
matched = pd.read_csv('matched.csv')

In [19]:
matched.head()

,Date,city,Tour,lat,lng
0,12/11/2014,Wallingford,The 1975 World Tour,51.600,-1.125
1,12/13/2014,Stroudsburg,The 1975 World Tour,40.986,-75.195
2,12/19/2014,Lake Buena Vista,The 1975 World Tour,28.375,-81.520
3,11/21/2015,Southampton,I Like It When You Sleep Tour,50.909,-1.404
4,11/23/2015,Southend On Sea,I Like It When You Sleep Tour,51.538,0.713


In [20]:
combined = pd.concat([sound, matched], ignore_index=True)

In [21]:
combined.shape

(719, 7)

In [22]:
combined.shape

(719, 7)

In [23]:
combined['lat'].isna().sum()

np.int64(62)

In [24]:
combined = combined.dropna(subset=['lat'])

In [25]:
combined.head()

,Date,city,Tour,country,lat,lng,population
0,9/5/2014,Kansas City,The 1975 World Tour,United States,39.1238,-94.5541,1714910.0
1,9/6/2014,St. Louis,The 1975 World Tour,United States,38.6359,-90.2451,2129918.0
2,9/7/2014,Boston,The 1975 World Tour,United States,42.3188,-71.0852,4453352.0
3,9/14/2014,Shrewsbury,The 1975 World Tour,United Kingdom,52.7080,-2.7540,71715.0
4,9/21/2014,Manchester,The 1975 World Tour,United Kingdom,53.4790,-2.2452,547627.0


In [26]:
combined["Date"] = pd.to_datetime(combined["Date"], format="%m/%d/%Y")

In [27]:
combined = combined.drop(columns={'population'})

In [28]:
combined.to_csv('combined.csv')

In [ ]:
import geopandas as gpd

gdf = gpd.GeoDataFrame(
    combined,
    geometry=gpd.points_from_xy(combined["lng"], combined["lat"]),
    crs="EPSG:4326",
)

In [30]:
gdf.explore()

In [2]:
import pandas as pd
import geopandas as gpd
fixed = pd.read_csv('combined_fixed.csv')

In [3]:
fixedgdf = gpd.GeoDataFrame(
    fixed,
    geometry=gpd.points_from_xy(fixed["lng"], fixed["lat"]),
    crs="EPSG:4326",
)

In [8]:
fixedgdf.explore(tiles='cartodb positron')

In [4]:
fixedgdf["city"] = fixedgdf["city"].replace("Köln", "Cologne")
fixedgdf["city"] = fixedgdf["city"].replace("St. Pauls", "St. Paul")
fixedgdf["city"] = fixedgdf["city"].replace("Scheessel", "Scheeßel")

In [5]:
fixedgdf.to_file('fixedgdf.geojson', driver='GeoJSON')